# Predicting next-month arrears
Follows `protocol.yaml`: temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, selection by mean CV ROC AUC, one holdout score.

**Leakage decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month (a consequence of the arrears), so it is not known at prediction time and is excluded. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
train_mask = ~df.index.isin(hold_idx)
train, hold = df[train_mask], df[~train_mask]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())

In [ ]:
TARGET='arrears_next_month'
NUM=['occupants','kwh','bill_eur','late_payments_12m']
CAT=['tariff']
FEATURES=NUM+CAT   # final_notice_sent excluded: recorded after the outcome
def prep(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
# all candidates named before any scoring
candidates = {
 'baseline_prior': lambda: make_pipeline(prep(False), DummyClassifier(strategy='prior')),
 'logreg_C0.1': lambda: make_pipeline(prep(True), LogisticRegression(C=0.1, max_iter=1000)),
 'logreg_C1': lambda: make_pipeline(prep(True), LogisticRegression(C=1.0, max_iter=1000)),
 'hgb_depth3': lambda: make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
 'rf_leaf20': lambda: make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=0)),
}
print(list(candidates))

In [ ]:
# walk-forward CV over whole billing months within training rows: 5 splits
months = np.sort(train.billing_month.unique())
n=len(months); k=5; size=n//(k+1)
folds=[]
for i in range(k):
    tr_m = months[:size*(i+1)]
    va_m = months[size*(i+1): size*(i+2) if i<k-1 else n]
    folds.append((np.where(train.billing_month.isin(tr_m))[0], np.where(train.billing_month.isin(va_m))[0]))
for tr,va in folds: print(len(tr),len(va))
Xtr, ytr = train[FEATURES], train[TARGET].to_numpy()
cv_scores={}
for name,mk in candidates.items():
    s=[]
    for tr,va in folds:
        m=mk().fit(Xtr.iloc[tr], ytr[tr])   # preprocessing is inside the pipeline, fitted per fold
        s.append(float(roc_auc_score(ytr[va], m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name]=s
cv_table=pd.DataFrame({n:{'mean':np.mean(s),'std':np.std(s)} for n,s in cv_scores.items()}).T
print(cv_table)

In [ ]:
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
final = candidates[chosen]().fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print(chosen, 'holdout roc_auc =', round(holdout_auc,4))

In [ ]:
train_pos = df.index[train_mask]   # 0-based row positions in data.csv
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(train_pos[i]) for i in tr], [int(train_pos[i]) for i in va]] for tr,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_idx],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))